In [1]:
from parachorpy import parachor as IFT
import numpy as np, pandas as pd, csv, os
from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary
import matplotlib.pyplot as plt
from CNT_BINARY import SHLMixture, SCLMixture
from math import isnan

In [2]:
if __name__ == '__main__':
    
    json_files  = ["Mulero_2012.json"]
    model       = IFT.InterfacialTension(json_files)
    model.ctREFPROP_init('~/Software/REFPROP_BETA/REFPROP-cmake/build', gerg_enable=1)

    MIXTURE   = "CO2;Argon"
    z         = [0.95, 0.05]
    T_list    = np.arange(220, 300, 1).tolist()
    kij       = 0.0
    phi_ij    = 1.0
        
    
results_all = []

for T in T_list:
    result = model.TP_DIAGRAM(MIXTURE, z, T, verbose=False)

    # tiny tolerance to avoid numerical issues exactly at phase limits
    tol_frac = 0.000005
    P_bub = result['bubble']['P_bar']
    P_dew = result['dew']['P_bar']

    # evaluate at both endpoints (tweaked by tol)
    PRESSURES = np.array([
        P_bub * (1.0 - tol_frac),  # bubble-side
        P_dew * (1.0 + tol_frac),  # dew-side
    ])

    print(f"T={T:.2f} K | evaluating at P_bubble={PRESSURES[0]:.6f} bar, P_dew={PRESSURES[1]:.6f} bar")

    Pbar, gamma_Parachor, gamma_WSD, x, y, rhol, rhov, mml, mmv = \
        model.gamma_TP(MIXTURE, z, T, PRESSURES, kij, phi_ij)

    
    # per-temperature envelope snapshot
    envelope = {
        "bubble": {
            "P_bar": round(P_bub, 6),
            "x": [round(v, 6) for v in result['bubble']['x']],
            "y": [round(v, 6) for v in result['bubble']['y']],
        },
        "dew": {
            "P_bar": round(P_dew, 6),
            "x": [round(v, 6) for v in result['dew']['x']],
            "y": [round(v, 6) for v in result['dew']['y']],
        },
        "Tc_K": round(result["Tc"], 4),
    }

    # build labeled rows for the two pressures
    samples = []
    for P, s1, s2, xl, yv, ml, mv, rhol_i, rhog_i in zip(Pbar, gamma_Parachor, gamma_WSD, x, y, mml, mmv, rhol, rhov):
        # decide which endpoint this is closest to
        point_type = "bubble" if abs(P - P_bub) <= abs(P - P_dew) else "dew"

        samples.append({
            "point_type": point_type,                      # "bubble" or "dew"
            "T_K": round(T, 4),
            "P_bar": round(P, 6),
            "gamma_Parachor_mN_m": round(s1, 8),
            "gamma_WSD_mN_m": round(s2, 8),
            "x": [round(v, 6) for v in xl],
            "y": [round(v, 6) for v in yv],
            "M_liq_kg_per_mol": round(ml, 8),
            "M_vap_kg_per_mol": round(mv, 8),
            "rho_liq_kg_per_m3": round(rhol_i, 6),
            "rho_vap_kg_per_m3": round(rhog_i, 6),
        })

    # one record per temperature with both labeled points
    results_all.append({
        "mixture": MIXTURE,
        "z": z,
        "kij": kij,
        "phi_ij": phi_ij,
        "T_K": round(T, 4),
        "envelope": envelope,   # bubble/dew data & Tc for this T
        "samples": samples,     # two rows: bubble & dew
    })

    # (optional) your printout
    print("\n--- Results ---")
    print(f"Temperature           : {T:.4f} K")
    print(f"Critical Temperature  : {envelope['Tc_K']:.4f} K")
    print("Pbar:", [round(float(v), 6) for v in Pbar[:2]])
    print("gamma_Parachor_mN_m:", [round(float(v), 8) for v in gamma_Parachor[:2]])
    print("gamma_WSD_mN_m:", [round(float(v), 8) for v in gamma_WSD[:2]])
    print("x:", [[round(float(e), 6) for e in arr] for arr in x[:2]])
    print("y:", [[round(float(e), 6) for e in arr] for arr in y[:2]])
    print("M_liq:", [round(float(v), 8) for v in mml[:2]])
    print("M_vap:", [round(float(v), 8) for v in mmv[:2]])
    print("rho_liq:", [round(float(v), 6) for v in rhol[:2]])
    print("rho_vap:", [round(float(v), 6) for v in rhov[:2]])
    print("--- End ---\n")

REFPROP version = 10.0.2.01
GERG-2008 model = Enabled
T=220.00 K | evaluating at P_bubble=32.899371 bar, P_dew=6.337688 bar
Given temperature is greater than Tc (150.687) for Argon
Computing interfacial tension at T = 0.9*150.687 for Parachor method

--- Results ---
Temperature           : 220.0000 K
Critical Temperature  : 304.1282 K
Pbar: [32.899371, 6.337688]
gamma_Parachor_mN_m: [13.29679966, 15.65777327]
gamma_WSD_mN_m: [15.76255928, 15.67420404]
x: [[1.0, 0.0], [0.999385, 0.000615]]
y: [[0.218852, 0.781148], [0.949996, 0.050004]]
M_liq: [44.0095, 44.00700084]
M_vap: [40.83686543, 43.80640772]
rho_liq: [1172.045168, 1165.945695]
rho_vap: [83.617556, 16.608613]
--- End ---

T=221.00 K | evaluating at P_bubble=33.064936 bar, P_dew=6.608932 bar
Given temperature is greater than Tc (150.687) for Argon
Computing interfacial tension at T = 0.9*150.687 for Parachor method

--- Results ---
Temperature           : 221.0000 K
Critical Temperature  : 304.1282 K
Pbar: [33.064936, 6.608932]
ga

In [3]:
# --- choose which surface tension to use for SHL ---
GAMMA_KEY = "gamma_Parachor_mN_m"  # or "gamma_WSD_mN_m"

# --- choose which side to compute ---
SIDE = "dew"      # "bubble", "dew", or "both"
# -----------------------------------

def collect_shl_points(results_all, side):
    points = []
    for rec in results_all:
        T_K = rec["T_K"]
        keyed = {s["point_type"]: s for s in rec["samples"]}

        s = keyed.get(side)
        if s is None:
            print(f"[skip] T={T_K:.2f} K | missing {side} sample")
            continue

        Psat_Pa   = s["P_bar"] * 1e5
        gamma_N_m = s[GAMMA_KEY] * 1e-3
        rho_L     = s["rho_liq_kg_per_m3"]
        x_list    = s["x"]

        # Basic sanity checks
        if gamma_N_m <= 0.0:
            print(f"[skip] T={T_K:.2f} K, {side:5s} | gamma<=0 from {GAMMA_KEY} (γ={gamma_N_m:.3e} N/m) — near Tc?")
            continue
        if rho_L <= 0.0:
            print(f"[skip] T={T_K:.2f} K, {side:5s} | rho_L invalid ({rho_L:g})")
            continue
        if abs(sum(x_list) - 1.0) > 1e-8:
            print(f"[skip] T={T_K:.2f} K, {side:5s} | x does not sum to 1 (sum={sum(x_list):.6f})")
            continue

        points.append({
            "T_K": T_K,
            "Psat_Pa": Psat_Pa,
            "gamma_N_m": gamma_N_m,
            "rho_L": rho_L,
            "x_list": x_list,
            "meta": {"point_type": side, "P_bar": s["P_bar"], "y_list": s["y"]},
        })
    return points

# --- run SHL only on valid points; skip those failing CNT conditions ---
M_CO2, M_Ar = 44.00985e-3, 39.948e-3
M_list = [M_CO2, M_Ar]
Jcrit = 1e12

def run_shl(points, label):
    results = []
    for row in points:
        try:
            shl = SHLMixture(
                Jcrit=Jcrit,
                Psat=row["Psat_Pa"],
                T=row["T_K"],
                gamma=row["gamma_N_m"],
                rho_L=row["rho_L"],
                M_list=M_list,
                x_list=row["x_list"],
            )
            DeltaP, Pc, rc = shl.compute()
            results.append({"T_K": row["T_K"], "Pc_bar": Pc/1e5, "DeltaP_Pa": DeltaP, "rc_m": rc})
        except ValueError as e:
            print(f"[skip] T={row['T_K']:.2f} K, {label:5s} | {e}")
            continue
    return results

if SIDE in ("bubble", "dew"):
    shl_points   = collect_shl_points(results_all, SIDE)
    shl_results  = run_shl(shl_points, SIDE)
    csv_filename = f"SHL_CO2_Ar_{SIDE}.csv"

    with open(csv_filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["T_K", "Pc_bar"])
        for r in shl_results:
            print(f"T={r['T_K']:.2f} K | SHL {SIDE} Pc = {r['Pc_bar']:.3f} bar")
            writer.writerow([r["T_K"], r["Pc_bar"]])

elif SIDE == "both":
    for side in ("bubble", "dew"):
        shl_points  = collect_shl_points(results_all, side)
        shl_results = run_shl(shl_points, side)
        csv_filename = f"SHL_CO2_Ar_{side}.csv"

        with open(csv_filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["T_K", "Pc_bar"])
            for r in shl_results:
                print(f"T={r['T_K']:.2f} K | SHL {side} Pc = {r['Pc_bar']:.3f} bar")
                writer.writerow([r["T_K"], r["Pc_bar"]])
else:
    raise ValueError("SIDE must be 'bubble', 'dew', or 'both'.")

[skip] T=294.00 K, dew   | gamma<=0 from gamma_Parachor_mN_m (γ=0.000e+00 N/m) — near Tc?
[skip] T=295.00 K, dew   | gamma<=0 from gamma_Parachor_mN_m (γ=0.000e+00 N/m) — near Tc?
T=220.00 K | SHL dew Pc = -175.640 bar
T=221.00 K | SHL dew Pc = -170.895 bar
T=222.00 K | SHL dew Pc = -166.204 bar
T=223.00 K | SHL dew Pc = -161.567 bar
T=224.00 K | SHL dew Pc = -156.983 bar
T=225.00 K | SHL dew Pc = -152.452 bar
T=226.00 K | SHL dew Pc = -147.973 bar
T=227.00 K | SHL dew Pc = -143.545 bar
T=228.00 K | SHL dew Pc = -139.168 bar
T=229.00 K | SHL dew Pc = -134.841 bar
T=230.00 K | SHL dew Pc = -130.564 bar
T=231.00 K | SHL dew Pc = -126.336 bar
T=232.00 K | SHL dew Pc = -122.157 bar
T=233.00 K | SHL dew Pc = -118.026 bar
T=234.00 K | SHL dew Pc = -113.943 bar
T=235.00 K | SHL dew Pc = -109.906 bar
T=236.00 K | SHL dew Pc = -105.917 bar
T=237.00 K | SHL dew Pc = -101.973 bar
T=238.00 K | SHL dew Pc = -98.075 bar
T=239.00 K | SHL dew Pc = -94.222 bar
T=240.00 K | SHL dew Pc = -90.413 bar
T=24

In [4]:
# --- choose which surface tension to use ---
GAMMA_KEY = "gamma_Parachor_mN_m"  # or "gamma_WSD_mN_m"

# --- choose which side to compute ---
SIDE = "bubble"      # <<< change to "bubble" or "dew" here
# --------------------------------------

scl_points = []

for rec in results_all:
    T_K = rec["T_K"]
    keyed = {s["point_type"]: s for s in rec["samples"]}

    # pick only the chosen side
    s = keyed.get(SIDE)
    if s is None:
        print(f"[skip] T={T_K:.2f} K | missing {SIDE} sample")
        continue

    # Extract needed fields
    Psat_Pa   = s["P_bar"] * 1e5
    gamma_N_m = s[GAMMA_KEY] * 1e-3
    rho_L     = s["rho_liq_kg_per_m3"]
    rho_g     = s["rho_vap_kg_per_m3"]
    x_list    = s["x"]
    y_list    = s["y"]

    # Sanity checks
    if gamma_N_m <= 0.0:
        print(f"[skip] T={T_K:.2f} K | gamma<=0 ({gamma_N_m:.3e})")
        continue
    if rho_L <= 0.0 or rho_g <= 0.0:
        print(f"[skip] T={T_K:.2f} K | invalid densities")
        continue
    if abs(sum(x_list) - 1.0) > 1e-8 or abs(sum(y_list) - 1.0) > 1e-8:
        print(f"[skip] T={T_K:.2f} K | composition sum error")
        continue

    scl_points.append({
        "T_K": T_K,
        "Psat_Pa": Psat_Pa,
        "gamma_N_m": gamma_N_m,
        "rho_L": rho_L,
        "rho_g": rho_g,
        "x_list": x_list,
        "y_list": y_list,
        "meta": {"point_type": SIDE, "P_bar": s["P_bar"]},
    })

# --- constants ---
M_CO2, M_Ar = 44.00985e-3, 39.948e-3
M_list = [M_CO2, M_Ar]
Jcrit = 1e12

# --- run SCL ---
def run_scl(points, label):
    results = []
    for row in points:
        try:
            scl = SCLMixture(
                Jcrit=Jcrit,
                Psat=row["Psat_Pa"],
                T=row["T_K"],
                gamma=row["gamma_N_m"],
                rho_L=row["rho_L"],
                rho_g=row["rho_g"],
                M_list=M_list,
                x_list=row["x_list"],
                y_list=row["y_list"],
            )
            Pc, rc, lnS = scl.compute()
            results.append({"T_K": row["T_K"], "Pc_bar": Pc/1e5, "rc_m": rc, "lnS": lnS})
        except ValueError as e:
            print(f"[skip] T={row['T_K']:.2f} K, {label:5s} | {e}")
            continue
    return results

scl_results = run_scl(scl_points, SIDE)

# --- save results ---
import csv
csv_filename = f"SCL_CO2_Ar_{SIDE}.csv"

with open(csv_filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["T_K", "Pc_bar"])  # header row
    for r in scl_results:
        print(f"T={r['T_K']:.2f} K | SCL {SIDE} Pc = {r['Pc_bar']:.3f} bar")
        writer.writerow([r["T_K"], r["Pc_bar"]])


[skip] T=264.00 K | gamma<=0 (0.000e+00)
[skip] T=265.00 K | gamma<=0 (0.000e+00)
[skip] T=266.00 K | gamma<=0 (0.000e+00)
[skip] T=267.00 K | gamma<=0 (0.000e+00)
[skip] T=268.00 K | gamma<=0 (0.000e+00)
[skip] T=269.00 K | gamma<=0 (0.000e+00)
[skip] T=270.00 K | gamma<=0 (0.000e+00)
[skip] T=287.00 K | gamma<=0 (0.000e+00)
[skip] T=288.00 K | gamma<=0 (0.000e+00)
[skip] T=289.00 K | gamma<=0 (0.000e+00)
[skip] T=290.00 K | gamma<=0 (0.000e+00)
[skip] T=291.00 K | gamma<=0 (0.000e+00)
[skip] T=292.00 K | gamma<=0 (0.000e+00)
[skip] T=293.00 K | gamma<=0 (0.000e+00)
[skip] T=294.00 K | gamma<=0 (0.000e+00)
[skip] T=295.00 K | gamma<=0 (0.000e+00)
[skip] T=296.00 K | gamma<=0 (0.000e+00)
[skip] T=297.00 K | gamma<=0 (0.000e+00)
T=220.00 K | SCL bubble Pc = 44.629 bar
T=221.00 K | SCL bubble Pc = 44.519 bar
T=222.00 K | SCL bubble Pc = 44.425 bar
T=223.00 K | SCL bubble Pc = 44.347 bar
T=224.00 K | SCL bubble Pc = 44.285 bar
T=225.00 K | SCL bubble Pc = 44.238 bar
T=226.00 K | SCL bubbl